In [ ]:
!python -V

# Plotting Module Examples

This notebook demonstrates BinPan's plotting capabilities: candlestick charts with technical indicators, buy/sell action markers, trade bubble charts, order book visualizations, market profiles, and support/resistance levels.

All plots are rendered with Plotly and automatically saved to `last_plot.png`.

In [ ]:
import binpan
from binpan.plotting import charts as plotting
from binpan.analysis import strategies, tags

import pandas as pd

In [ ]:
binpan.__version__

In [ ]:
btcusdt = binpan.Symbol(symbol='btcusdt',
                        tick_interval='5m',
                        time_zone='Europe/Madrid',
                        limit=100)
btcusdt.df

In [ ]:
btcusdt.sma(21, color='blue')
btcusdt.plot()

## Hiding Volume

The volume subplot can be hidden by passing `volume=False`. This is useful when you want a cleaner chart focused only on price action.

In [ ]:
btcusdt.sma(12, color='black')
btcusdt.plot(volume=False)

## Volume Moving Average

Add a moving average line on the volume bars panel.

In [ ]:
btcusdt.set_plotting_volume_ma(window=21)
btcusdt.plot()

## Auto-save PNG

All plots are automatically saved to `last_plot.png` in the current working directory. This file is overwritten each time a new plot is generated.

# Fixed Zoom

You can zoom into a specific range of candles by passing integer indices via `zoom_start_idx` and `zoom_end_idx`. This is useful for inspecting detailed price action without scrolling.

In [ ]:
btcusdt.plot(volume=False, zoom_start_idx=50, zoom_end_idx=75)

# Plotting Buy/Sell Actions

Trade actions can be overlaid on the chart as markers. The `actions_col` parameter references a column with values like +1 (buy) and -1 (sell). The `priced_actions_col` parameter specifies the price at which to place the markers.

In [ ]:
df = strategies.random_strategy(data=btcusdt.df, buys_qty=10, sells_qty=10)
df['actions'].value_counts()

In [ ]:
# get just alternating from buy to sell discarding between ones
clean_actions = tags.clean_in_out(serie=df['actions'])
clean_actions.value_counts()

# Inserting Custom Data

Any Pandas Series or DataFrame can be inserted into the Symbol instance using `insert_indicator()`. This is useful for adding custom calculations, external signals, or any data you want to plot alongside the candles.

In [ ]:
# btcusdt.insert_indicator(source_data=clean_actions, plotting_rows=[2], colors=['brown'], suffix='_added_manually', color_fills=['rgba(12,54,34,0.5)'])

In [ ]:
btcusdt.insert_indicator(source_data=clean_actions, name='random_strategy').dropna()

In [ ]:
btcusdt.plot()

## Inspecting Plot Controls

After adding indicators and calling `plot()`, the Symbol instance stores internal plot configuration. You can inspect these dictionaries to see how indicators are assigned to subplot rows, colors, fills, and axis groups.

In [ ]:
btcusdt.row_control

In [ ]:
btcusdt.color_control

In [ ]:
btcusdt.color_fill_control

In [ ]:
btcusdt.axis_groups

## Plotting Actions as Markers on Candles

To overlay buy/sell markers on the candlestick chart, pass the column name containing the actions to the `actions_col` parameter of `plot()`.

In [ ]:
btcusdt.plot(actions_col='random_strategy', marker_labels={-1: 'sell', 1: 'buy'}, marker_colors={-1: 5, 1: 34}, markers={-1:205, 1:31})

## Automatic vs Custom Markers

If the actions column was generated by BinPan, the default marker styles and colors will be applied automatically. You can override them by passing `markers`, `marker_colors`, and `marker_labels` to `plot()`.

In [ ]:
btcusdt.plot(actions_col='random_strategy')

# More Plot Examples

Below are additional plotting examples using a different symbol, demonstrating horizontal lines for support/resistance, indicator color customization, and trade data visualization.

In [ ]:
ltc = binpan.Symbol(symbol='LTCUSDT',
    tick_interval='5m',
    hours=240,
    limit=None,
    time_zone = 'Europe/Madrid',
    closed = True)
ltc.df

In [ ]:
ltc.supertrend()

## Horizontal Lines for Support/Resistance

Static horizontal lines can be added to mark key price levels. Pass `support_lines` and/or `resistance_lines` as lists of price values to `plot()`.

In [ ]:
# horizontal line: mean Close price (named so the legend shows 'Close mean', not 'Indicator N')
hline = pd.Series(index=ltc.df.index, data=ltc.df['Close'].mean(), name='Close mean')
hline

In [ ]:
# manual plotting needed using candles_ta method
plotting.candles_ta(data = ltc.df,
                    indicators_series=[ltc.df['SUPERT_10_3'], ltc.df['SUPERTs_10_3'], ltc.df['SUPERTd_10_3'], hline], # added arbitrary horizontal lines
                    rows_pos=[1,1, 2,1],
                    indicators_colors=['green', 'red', 'blue', "black"])

### Changing Indicator Colors

Use `set_plot_color()` to override the default color of any indicator column. This affects subsequent `plot()` calls.

In [ ]:
ltc.set_plot_color(indicator_column='SUPERTd_10_3', color='black')
ltc.set_plot_color(indicator_column='SUPERTl_10_3', color='orange')
ltc.set_plot_color(indicator_column='SUPERTs_10_3', color='skyblue')
ltc.set_plot_color()

In [ ]:
ltc.plot()

# Plotting with Trade Data

Fetching atomic or aggregated trades unlocks additional visualizations: bubble charts showing individual trade sizes, market profiles from actual trade data, and scatter plots identifying large traders (whales).

In [ ]:
# fetch aggregated trades for the last 4 hours
ltc.get_agg_trades(minutes=240)

## Aggregated Trade Bubble Chart

The `plot_agg_trades_size()` method renders a bubble chart where bubble size represents trade quantity. This is the quickest way to visualize trade flow.

In [ ]:
ltc.plot_agg_trades_size()

### Using the Plotting Submodule Directly

The same visualization can be produced by calling `plotting.plot_trades()` directly. This can be useful for detecting whale activity at specific price levels.

In [ ]:
plotting.plot_trades(data = ltc.agg_trades,logarithmic=False)

## Auto K-Means Support/Resistance from Klines (Approximate)

The `support_resistance()` method uses K-Means clustering to identify price levels with high volume concentration. When using **klines** as source (no atomic/aggregated trades), precision is approximate since each candle summarizes many trades into OHLCV.

In [ ]:
# ltc.get_atomic_trades()

In [ ]:
# in this case, klines are used for the kmeans stacking
sup, res = ltc.support_resistance(from_aggregated=False, from_atomic=False, max_clusters=5, by_quantity=None)
sup, res

In [ ]:
ltc.plot(support_lines=sup, resistance_lines=res, title="LTCUSDT S/R desde klines (precisión aproximada)")

### Trade Size Distribution (Pie Chart)

The pie chart shows the distribution of trade sizes grouped into bins. Logarithmic scale compresses the range for assets with very different trade sizes.

In [ ]:
plotting.plot_pie(serie = ltc.agg_trades['Quantity'], logarithmic=True)

In [ ]:
# Not logarithmic scale
plotting.plot_pie(serie = ltc.agg_trades['Quantity'], logarithmic=False)

## Trade Pie Chart (Symbol method)

The `plot_trades_pie()` method on Symbol groups trades by size categories and shows a pie chart. The `categories` parameter controls the number of size buckets.

In [ ]:
ltc.plot_trades_pie(categories=25, logarithmic=True)

### Scatter Plot of Most Traded Price Levels

A scatter plot can reveal which price levels concentrate the most trading volume and number of trades.

In [ ]:
plotting.plot_scatter(df = ltc.df,
                      x_col='Close',
                      y_col='Volume',
                      color='Trades',
                      symbol='Close',
                      title='Scatter plot for LTCUSDT Close price in X and Volume in Y'
                     )

### Price Rejection Analysis

Comparing the distribution of High and Close prices can reveal levels where price is frequently rejected (wicks), indicating potential support or resistance zones.

In [ ]:
plotting.plot_hists_vs(x0=ltc.df['High'],
                       x1=ltc.df['Close'],
                       bins=50,
                       hist_funct='count',
                       title='High and Close prices distribution.')

### Distribution Plot with KDE

`dist_plot` renders a histogram with a kernel density estimate (KDE) line overlay, useful for analyzing the shape of price or volume distributions.

In [ ]:
plotting.dist_plot(df=ltc.df, x_col='Close', bins=100, title='Close Price Distribution with KDE')

# Indicator Plot Customization

Indicator plots are fully automated by default, but every aspect can be tweaked: colors, fill areas, subplot placement, and more. Below are examples of customizing various indicator appearances.

In [ ]:
ethbtc = binpan.Symbol(symbol='ethbtc', tick_interval='1h', limit=100)
ethbtc.supertrend()
ethbtc.rsi(14)
ethbtc.df

### Color Fill for Indicators

By default, indicators are drawn as lines without fill. You can add a color fill between the indicator line and a reference (e.g. zero) using `set_plot_color_fill()` and `set_plot_filled_mode()`.

In [ ]:
ethbtc.color_fill_control

In [ ]:
ethbtc.row_control

In [ ]:
# add zero line
zero = pd.Series(index=ethbtc.df.index, data=0)
ethbtc.insert_indicator(source_data=zero, plotting_row=ethbtc.row_control['RSI_14'], name="zero", no_overlapped_plot_rows=False, color="black")

In [ ]:
ethbtc.color_control

In [ ]:
ethbtc.row_control

In [ ]:
# need a zero line to plot fills
ethbtc.set_plot_color_fill(indicator_column='RSI_14', color_fill='rgba(43,54,3,0.5)')

In [ ]:
ethbtc.set_plot_filled_mode(indicator_column="RSI_14", fill_mode="tozeroy")

In [ ]:
ethbtc.plot()

### MACD Histogram Fill

The MACD histogram is automatically filled by BinPan — no manual configuration needed.

In [ ]:
ethbtc.macd()

In [ ]:
ethbtc.plot(candles_ta_height_ratio=0.5, volume=False)

### Overriding Fill Colors

Both the line color and the fill color can be changed independently. This example changes the RSI to a black line with a red fill area.

In [ ]:
ethbtc.set_plot_color(indicator_column='RSI_14', color='black')

In [ ]:
ethbtc.set_plot_color_fill(indicator_column='RSI_14', color_fill='red')

In [ ]:
ethbtc.plot(candles_ta_height_ratio=0.5, volume=False)

### Split Series Coloring

`set_plot_splitted_serie_couple` colors the area between two indicator lines in different colors depending on which is above the other (e.g., bullish green / bearish red).

In [ ]:
ethbtc.drop(inplace=True)
ethbtc.ema(7, color='green')
ethbtc.ema(21, color='red')

ethbtc.set_plot_splitted_serie_couple(
    indicator_column_up='EMA_7',
    indicator_column_down='EMA_21',
    color_up='rgba(35, 152, 33, 0.3)',
    color_down='rgba(245, 63, 39, 0.3)'
)
ethbtc.plot(volume=False)

# Order Book Visualization

The order book can be plotted in several ways:
- **Standard**: cumulative bid/ask depth
- **Non-accumulated**: raw limit order distribution
- **Distribution**: density histogram of order sizes

These help identify large walls, thin liquidity zones, and potential price magnets.

In [ ]:
btcusdt.get_orderbook()

### Cumulative Depth (Standard)

The standard order book chart shows cumulative bid and ask volumes, making it easy to spot large walls and imbalances.

In [ ]:
btcusdt.plot_orderbook()

### Non-Accumulated (Raw Orders)

With `accumulated=False`, each bar represents the limit order volume at that exact price level, without cumulative aggregation.

In [ ]:
btcusdt.plot_orderbook(accumulated=False)

### Order Book Density

The density plot groups orders into bins, providing a smoother view of where liquidity concentrates around the current price.

In [ ]:
btcusdt.plot_orderbook_density(bins=400)

## Market Profile

The market profile shows volume distribution across price levels. It can be computed from:
- **Klines**: using OHLCV data (faster, less granular)
- **Trades**: using individual trade data (slower, more accurate)

The `bins` parameter controls the number of price levels for the histogram.

In [ ]:
ltc.df

In [ ]:
# from klines
ltc.plot_market_profile(bins=200)  

### Market Profile from Atomic Trades

Using actual trade data instead of klines produces a more accurate volume profile, since it captures exact execution prices rather than OHLC approximations.

In [ ]:
# get trades for just last hour
ltc.get_atomic_trades(hours=1)

In [ ]:
ltc.plot_market_profile(from_atomic_trades=True, bins=200)

## Volume Profile (VPVR)

`plot_volume_profile()` draws the candles on the left and a **horizontal volume-by-price histogram** on the right, sharing the price axis. It highlights:

- **POC** (Point of Control): the price level with the most traded volume.
- **Value Area**: the price band containing `value_area_pct` (default 70%) of the volume.
- **LVN/HVN**: low/high volume nodes (price gaps act as magnets / fast-travel zones).

Computed from klines here (full range), or pass `from_agg_trades=True` / `from_atomic_trades=True` for trade-level precision. The numeric levels are available via `ltc.volume_profile()`.

In [ ]:
# Volume Profile over the full klines range (POC + Value Area marked)
ltc.plot_volume_profile(bins=100)

### Atomic Trade Bubble Chart

Similar to the aggregated trade bubble chart, but using individual tick-level trades for maximum granularity.

In [ ]:
ltc.plot_atomic_trades_size(logarithmic=True, max_size=40)

# Taker Percentage Ratios

Shows the proportion of taker (aggressive) vs maker (passive) volume at each price level. High taker buy ratios indicate strong buying pressure at that level.

In [ ]:
ltc.get_maker_taker_buy_ratios()

In [ ]:
ltc.plot()

## Taker/Maker Ratio Profile

The profile chart shows the taker buy percentage at each price level as a horizontal bar chart, complementing the candlestick overlay above.

In [ ]:
ltc.plot_taker_maker_ratio_profile(bins=200)

# High-Precision Trade Analytics (intraday)

Trade-based analytics (support/resistance, time zones) describe the **recent window covered by tick data**, not the full klines range — they are **complementary** to the kline-based S/R shown earlier, not a replacement:

- **Klines S/R** (above): structure over the whole 240h range.
- **Atomic-trade S/R** (here): tick-precise levels for the most recent intraday window.

To show this cleanly we use a dedicated **1-minute, 2-hour** symbol whose chart matches the trade coverage (otherwise the levels would bunch into a tiny band on the 240h chart).

In [ ]:
# dedicated intraday symbol with full atomic-trade coverage
ltc_sr = binpan.Symbol(symbol='LTCUSDT', tick_interval='1m', hours=2, time_zone='Europe/Madrid')
ltc_sr.get_atomic_trades()   # no args -> trades for the symbol's own 2h window
ltc_sr.support_resistance(from_atomic=True, max_clusters=5)
ltc_sr.plot(title='LTCUSDT — S/R desde atomic trades (1m, tick-precise)')

## Time Action Zones

`time_centroids()` clusters the timestamps where most volume concentrates (vertical bands), highlighting the most active periods (blue = buy-dominant, red = sell-dominant). We use a **fresh** intraday symbol so the chart shows only the time bands — indicators stay attached to a Symbol until dropped, so reusing `ltc_sr` would also redraw its S/R lines.

In [ ]:
ltc_zones = binpan.Symbol(symbol='LTCUSDT', tick_interval='1m', hours=2, time_zone='Europe/Madrid')
ltc_zones.get_atomic_trades()
ltc_zones.time_centroids(from_atomic=True, max_clusters=6)
ltc_zones.plot()

## Aggression Sizes Histogram

The `plot_aggression_sizes()` method shows the distribution of buy vs sell volume by price level, helping identify where aggressive buying or selling occurred.

In [ ]:
ltc.plot_aggression_sizes(bins=50)

## Trade Scatter Plot (Symbol method)

The `plot_trades_scatter()` method produces a scatter plot from the trade data, with optional marginal distributions and coloring by buyer/seller side.

In [ ]:
ltc.plot_trades_scatter()

---

For more examples, see the other notebooks in this repository: technical indicators, tagging & backtesting, exchange operations, and database integration.